# Lesson 26: Generative models: autoencoders, VAEs, GANs, flows and diffusion

*Companion notebook for [the lesson page](../lessons/26-generative-models.html). Worked solutions are in `solutions/26-generative-models-solutions.ipynb`; try each exercise first.*

## Setup

Install what this lesson needs. Skip this cell if your environment already has these packages. Everything runs offline on a CPU.

In [ ]:
%pip install -q numpy scipy matplotlib scikit-learn torch

Plotting helpers used throughout the notebook.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def show_digits(images, n=20, n_cols=10, title=None):
    """Draw up to n 8x8 digit images in a grid."""
    images = np.clip(np.asarray(images)[:n], 0, 1)
    n_rows = int(np.ceil(len(images) / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 0.8, n_rows * 0.85),
                             squeeze=False)
    for ax in axes.ravel():
        ax.axis("off")
    for ax, img in zip(axes.ravel(), images):
        ax.imshow(img.reshape(8, 8), cmap="gray_r", vmin=0, vmax=1)
    if title:
        fig.suptitle(title)
    plt.show()

def show_points(sets, labels, title=None, size=4.5):
    """Scatter several 2-D point sets on one square plot."""
    plt.figure(figsize=(size, size))
    for pts, label in zip(sets, labels):
        pts = np.asarray(pts)
        plt.scatter(pts[:, 0], pts[:, 1], s=3, alpha=0.5, label=label)
    plt.legend(markerscale=4)
    plt.axis("equal")
    if title:
        plt.title(title)
    plt.show()

## What a generative model is

A classifier learns $p(y \mid x)$; a **generative model** learns $p(x)$ itself. With it you can do two jobs: **density estimation** (score how plausible an example is) and **sampling** (draw new examples). A Gaussian mixture does both.

In [ ]:
import numpy as np
from sklearn.datasets import make_moons
from sklearn.mixture import GaussianMixture

moons, _ = make_moons(2000, noise=0.05, random_state=0)
gmm = GaussianMixture(n_components=8, random_state=0).fit(moons[:1500])

# job 1, density: log p(x) for a point on a moon and a point far away
print(gmm.score_samples([[0.0, 1.0], [0.0, 3.0]]).round(1))
print(round(gmm.score(moons[1500:]), 3))   # mean log p(x) on held-out points

# job 2, sampling: new points that look like the data
new_points, _ = gmm.sample(3)
print(new_points.round(2))

In [ ]:
# the data (blue) and 1,000 new points sampled from the mixture (orange)
gmm_samples, _ = gmm.sample(1000)
show_points([moons, gmm_samples], ["data", "GMM samples"], "A Gaussian mixture as a generative model")

## The map of model families

| Family | Trained by | Sampling and likelihood | Typical use today |
|---|---|---|---|
| Autoregressive | Maximum likelihood, next token | One call per token; exact likelihood | Text, code; images and audio as tokens |
| VAE | Maximising the ELBO | One decoder pass; a bound only | Latents for other generators |
| GAN | A game against a discriminator | One pass; no likelihood | Fast synthesis, super-resolution, extra loss |
| Normalising flow | Exact maximum likelihood | One inverse pass; exact | Density estimation, scientific data |
| Diffusion and flow matching | Regressing noise or velocity | Tens of calls; likelihood costly | Images, video, audio |
| Energy-based | Lowering an energy on data | Slow iterative sampling | Mostly research |

## Autoencoders

### Compress and reconstruct

An encoder squeezes each digit into a short code, a decoder rebuilds it, and training minimises the reconstruction error. The digit classifier below acts as a rough **judge** of generated images: real test digits get an average confidence of about 0.9 and roughly 45 of each digit.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.datasets import load_digits
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

torch.set_num_threads(1)     # tiny models: one thread is usually fastest

digits = load_digits()
X = digits.data / 16.0       # 1,797 images of 8x8 pixels, scaled to [0, 1]
X_tr, X_te, y_tr, y_te = train_test_split(
    X, digits.target, test_size=0.25, random_state=0, stratify=digits.target)
Xtr = torch.tensor(X_tr, dtype=torch.float32)
Xte = torch.tensor(X_te, dtype=torch.float32)

# a classifier that will act as a judge of generated digits
judge = LogisticRegression(max_iter=2000).fit(X_tr, y_tr)

def judge_report(images):
    """Mean confidence of the judge, and how many images it calls each digit."""
    p = judge.predict_proba(np.clip(images, 0, 1))
    counts = np.bincount(p.argmax(1), minlength=10)
    return round(float(p.max(1).mean()), 2), counts.tolist()

print(round(judge.score(X_te, y_te), 3))
print(judge_report(X_te))

In [ ]:
class AutoEncoder(nn.Module):
    def __init__(self, d_latent=2):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(64, 128), nn.ReLU(),
                                     nn.Linear(128, d_latent))
        self.decoder = nn.Sequential(nn.Linear(d_latent, 128), nn.ReLU(),
                                     nn.Linear(128, 64), nn.Sigmoid())

    def forward(self, x):
        return self.decoder(self.encoder(x))

def fit(model, loss_fn, epochs=100, lr=2e-3, batch=128):
    """Minibatch training on the digit images; loss_fn(model, x) -> loss."""
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    for epoch in range(epochs):
        for idx in torch.randperm(len(Xtr)).split(batch):
            loss = loss_fn(model, Xtr[idx])
            opt.zero_grad(); loss.backward(); opt.step()
    return model

torch.manual_seed(0)
ae = fit(AutoEncoder(), lambda m, x: F.mse_loss(m(x), x))

from sklearn.decomposition import PCA
pca = PCA(n_components=2).fit(X_tr)           # the linear baseline
pca_recon = pca.inverse_transform(pca.transform(X_te))
with torch.no_grad():
    print("autoencoder", round(F.mse_loss(ae(Xte), Xte).item(), 4))
print("PCA        ", round(((pca_recon - X_te) ** 2).mean(), 4))

PCA (lesson 04) is the linear version of the same idea; the non-linear autoencoder reconstructs better with the same two numbers per image.

### The latent space

Plot the codes and you see a cluster per digit, with gaps between. Decoding random standard-normal codes shows why a plain autoencoder is not a generator: nothing tells you where the codes live.

In [ ]:
with torch.no_grad():
    Z = ae.encoder(Xtr)                 # the 2-D code of every training digit
print(Z.mean(0).numpy().round(2), Z.std(0).numpy().round(2))

# try to generate: decode random codes from a standard normal
torch.manual_seed(1)
with torch.no_grad():
    fake = ae.decoder(torch.randn(1000, 2)).numpy()
print(judge_report(fake))

In [ ]:
# the 2-D latent space, one colour per digit, then some decoded random codes
plt.figure(figsize=(5, 5))
plt.scatter(Z[:, 0], Z[:, 1], c=y_tr, cmap="tab10", s=5)
plt.colorbar(label="digit")
plt.title("Autoencoder codes of the training digits")
plt.show()
show_digits(fake, title="Decoded N(0, I) codes: few kinds of digit")

### Denoising autoencoders

Corrupt the input, score against the clean original. The denoiser learns which way leads back towards real data, which is the idea behind diffusion models in session 3.

In [ ]:
def denoising_loss(model, x, noise=0.3):
    noisy = x + noise * torch.randn_like(x)    # corrupt the input...
    return F.mse_loss(model(noisy), x)         # ...but score against the clean x

torch.manual_seed(0)
plain = fit(AutoEncoder(d_latent=16), lambda m, x: F.mse_loss(m(x), x))
dae = fit(AutoEncoder(d_latent=16), denoising_loss)

torch.manual_seed(2)
noisy_te = Xte + 0.3 * torch.randn_like(Xte)
with torch.no_grad():
    for name, m in [("plain", plain), ("denoising", dae)]:
        print(f"{name:9s} clean in: {F.mse_loss(m(Xte), Xte).item():.4f}  "
              f"noisy in: {F.mse_loss(m(noisy_te), Xte).item():.4f}")

In [ ]:
show_digits(noisy_te[:10].numpy(), n=10, title="noisy test digits")
with torch.no_grad():
    show_digits(plain(noisy_te[:10]).numpy(), n=10, title="plain autoencoder")
    show_digits(dae(noisy_te[:10]).numpy(), n=10, title="denoising autoencoder")

### Hands-on: turn the autoencoder into a generator

The autoencoder failed as a generator because we sampled codes from the wrong distribution. Give it the right one.

1. Fit a `GaussianMixture` with 10 components to the training codes `Z`.
2. Draw 1,000 codes from the mixture, decode them with `ae.decoder` and run `judge_report`.
3. Compare with the standard-normal codes above. You are done when every digit appears and the confidence rises clearly above 0.62.
4. Think about why this two-stage recipe gets harder as the code grows from 2 to 100 dimensions.

In [ ]:
# Your code here

## Variational autoencoders

### A latent-variable model

Draw a code $z \sim \mathcal{N}(0, I)$, decode it with $p_\theta(x \mid z)$. The likelihood $p_\theta(x) = \int p_\theta(x \mid z)\,p(z)\,dz$ is intractable, so an encoder $q_\phi(z \mid x)$ proposes likely codes.

### The ELBO, gently

$$\log p_\theta(x) \ge \mathbb{E}_{q_\phi(z\mid x)}\big[\log p_\theta(x \mid z)\big] - D_{\mathrm{KL}}\big(q_\phi(z \mid x)\,\|\,p(z)\big)$$

Reconstruction wants each code to pin down its image; the KL term pulls every code distribution towards $\mathcal{N}(0, I)$. For a Gaussian encoder, $D_{\mathrm{KL}} = \tfrac12 \sum_j (\mu_j^2 + \sigma_j^2 - \log\sigma_j^2 - 1)$.

### The reparameterisation trick

Write the sample as $z = \mu + \sigma \odot \varepsilon$ with $\varepsilon \sim \mathcal{N}(0, I)$, so gradients flow into $\mu$ and $\sigma$. Note the three bug-avoiding details: predict $\log\sigma^2$, return logits, and sum both loss terms over their dimensions before averaging over the batch.

In [ ]:
class VAE(nn.Module):
    def __init__(self, d_latent=2):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(64, 128), nn.ReLU(),
                                     nn.Linear(128, 2 * d_latent))  # mu, log var
        self.decoder = nn.Sequential(nn.Linear(d_latent, 128), nn.ReLU(),
                                     nn.Linear(128, 64))            # pixel logits

    def encode(self, x):
        mu, logvar = self.encoder(x).chunk(2, dim=1)
        return mu, logvar

    def forward(self, x):
        mu, logvar = self.encode(x)
        eps = torch.randn_like(mu)
        z = mu + torch.exp(0.5 * logvar) * eps     # the reparameterisation trick
        return self.decoder(z), mu, logvar

def elbo_terms(model, x):
    logits, mu, logvar = model(x)
    # reconstruction: -log p(x|z) under a Bernoulli per pixel, summed over pixels
    recon = F.binary_cross_entropy_with_logits(logits, x, reduction="sum")
    # KL(q(z|x) || N(0, I)) in closed form, summed over latent dimensions
    kl = 0.5 * (mu ** 2 + logvar.exp() - logvar - 1).sum()
    return recon / len(x), kl / len(x)          # both per image

def vae_loss(model, x, beta=1.0):
    recon, kl = elbo_terms(model, x)
    return recon + beta * kl                    # the negative ELBO when beta = 1

### Train, sample and interpolate

The codes now sit on the prior (mean about 0, spread about 1), so samples from $\mathcal{N}(0, I)$ cover every digit. They are blurry, which is why the judge's confidence did not rise.

In [ ]:
torch.manual_seed(0)
vae = fit(VAE(), vae_loss)

torch.manual_seed(1)
with torch.no_grad():
    recon, kl = elbo_terms(vae, Xte)
    print(f"test: reconstruction {recon:.1f}, KL {kl:.2f} nats per image")
    mu, _ = vae.encode(Xtr)
    print(mu.mean(0).numpy().round(2), mu.std(0).numpy().round(2))
    samples = torch.sigmoid(vae.decoder(torch.randn(1000, 2))).numpy()
print(judge_report(samples))

In [ ]:
# the VAE's codes now fill a round blob around the origin
plt.figure(figsize=(5, 5))
plt.scatter(mu[:, 0], mu[:, 1], c=y_tr, cmap="tab10", s=5)
plt.colorbar(label="digit")
plt.title("VAE code means of the training digits")
plt.show()
show_digits(samples, n=30, title="VAE samples from N(0, I)")

Because the latent space has no gaps, you can walk through it from a 0 to a 6:

In [ ]:
i, j = np.where(y_te == 0)[0][0], np.where(y_te == 6)[0][0]
with torch.no_grad():
    mu, _ = vae.encode(Xte[[i, j]])
    path = torch.stack([(1 - a) * mu[0] + a * mu[1]
                        for a in torch.linspace(0, 1, 8)])
    frames = torch.sigmoid(vae.decoder(path)).numpy()
print(judge.predict(frames))

In [ ]:
show_digits(frames, n=8, n_cols=8, title="From a 0 to a 6 in latent space")

### β-VAE and disentanglement

Weighting the KL term by $\beta > 1$ pushes towards disentangled latents at a cost in reconstruction. Push it too far and the KL term wins outright. Try it: with $\beta = 4$ the KL falls to zero, the decoder ignores $z$, and every sample is the same average blob (**posterior collapse**).

In [ ]:
torch.manual_seed(0)
vae4 = fit(VAE(), lambda m, x: vae_loss(m, x, beta=4.0))
torch.manual_seed(1)
with torch.no_grad():
    recon4, kl4 = elbo_terms(vae4, Xte)
    print(f"beta=4: reconstruction {recon4:.1f}, KL {kl4:.2f}")
    samples4 = torch.sigmoid(vae4.decoder(torch.randn(1000, 2))).numpy()
print(judge_report(samples4))

### VQ-VAE: discrete latents

Snap each encoder output to the nearest codebook vector; the image becomes a grid of integer tokens that a transformer (lesson 08) can model like text. The straight-through estimator passes gradients through the snap.

In [ ]:
def quantise(z, codebook):
    """Snap each row of z to its nearest codebook vector, as in a VQ-VAE."""
    idx = torch.cdist(z, codebook).argmin(dim=1)   # one integer token per row
    z_q = codebook[idx]
    vq_loss = (F.mse_loss(z_q, z.detach())          # pull codes to the encoder
               + 0.25 * F.mse_loss(z, z_q.detach()))  # commit encoder to code
    z_st = z + (z_q - z).detach()  # straight-through: value z_q, gradient to z
    return z_st, idx, vq_loss

torch.manual_seed(0)
codebook = torch.randn(16, 4, requires_grad=True)   # K = 16 codes of size 4
z = torch.randn(6, 4, requires_grad=True)           # six encoder outputs
z_q, idx, vq_loss = quantise(z, codebook)
print(idx)                     # the "image" is now six integer tokens
z_q.sum().backward()
print(z.grad[0])               # ones: the gradient passed straight through

## Generative adversarial networks

### The game

$$\min_G \max_D \; \mathbb{E}_{x \sim p_{\text{data}}}[\log D(x)] + \mathbb{E}_{z}[\log(1 - D(G(z)))]$$

The generator uses the **non-saturating** loss: maximise $\log D(G(z))$, which in code means scoring fakes against the "real" label.

### A GAN on eight Gaussians

A ring of eight tight Gaussian blobs makes failures easy to see. `mode_report` returns how many modes receive at least 1% of samples, and the share of samples within three standard deviations of a mode.

In [ ]:
angles = torch.arange(8) * 2 * torch.pi / 8
CENTRES = 2.0 * torch.stack([angles.cos(), angles.sin()], dim=1)   # (8, 2)

def sample_ring(n, std=0.05):
    return CENTRES[torch.randint(0, 8, (n,))] + std * torch.randn(n, 2)

def mode_report(x, std=0.05):
    """Modes that get at least 1% of samples, and the share of good samples."""
    d = torch.cdist(x, CENTRES)                  # distance to every centre
    good = d.min(1).values < 3 * std             # within 3 std of some mode
    counts = torch.bincount(d.argmin(1)[good], minlength=8)
    covered = int((counts >= 0.01 * len(x)).sum())
    return covered, round(good.float().mean().item(), 2)

def mlp(n_in, n_out, hidden=128, depth=2):
    layers, d = [], n_in
    for _ in range(depth):
        layers += [nn.Linear(d, hidden), nn.ReLU()]
        d = hidden
    return nn.Sequential(*layers, nn.Linear(d, n_out))

torch.manual_seed(0)
print(mode_report(sample_ring(1000)))   # real data: all 8 modes, 99% good

In [ ]:
def train_gan(steps=3000, batch=256, lr=1e-3, betas=(0.9, 0.999)):
    G, D = mlp(2, 2), mlp(2, 1)      # D returns a logit: "how real is this?"
    opt_g = torch.optim.Adam(G.parameters(), lr=lr, betas=betas)
    opt_d = torch.optim.Adam(D.parameters(), lr=lr, betas=betas)
    bce = nn.BCEWithLogitsLoss()
    real_label, fake_label = torch.ones(batch, 1), torch.zeros(batch, 1)
    for step in range(1, steps + 1):
        real, fake = sample_ring(batch), G(torch.randn(batch, 2))
        # 1. discriminator: push D(real) towards 1 and D(fake) towards 0
        d_loss = bce(D(real), real_label) + bce(D(fake.detach()), fake_label)
        opt_d.zero_grad(); d_loss.backward(); opt_d.step()
        # 2. generator (non-saturating loss): make D call the fakes real
        g_loss = bce(D(fake), real_label)
        opt_g.zero_grad(); g_loss.backward(); opt_g.step()
        if step % 500 == 0:
            with torch.no_grad():
                print(step, mode_report(G(torch.randn(1000, 2))))
    return G

torch.manual_seed(8)
G = train_gan()

In [ ]:
torch.manual_seed(0)
with torch.no_grad():
    gan_samples = G(torch.randn(1000, 2))
show_points([sample_ring(1000), gan_samples], ["data", "GAN samples"], "Mode collapse")

Sharp samples, but only five of eight modes: **mode collapse**. Change the seed and the details change. GAN training is sensitive.

### Making GANs train

- The non-saturating generator loss (used above).
- Adam with $\beta_1 = 0.5$, as recommended by DCGAN.
- Spectral normalisation of the discriminator: `nn.utils.parametrizations.spectral_norm`.
- Wasserstein critics with a gradient penalty (WGAN-GP).
- Separate learning rates for the two players (TTUR).

Watch samples, not losses: GAN losses do not fall steadily as the generator improves.

### Where GANs still win

Single-pass generation: real-time synthesis, super-resolution, and the adversarial loss inside other models, such as latent-diffusion decoders and SD-Turbo from lesson 23.

### Hands-on: stabilise the GAN

1. Rerun `train_gan` from the same seed (8) with `betas=(0.5, 0.999)` and compare the final `mode_report` with the collapsed run.
2. Write `train_wgan_gp`: a critic `C = mlp(2, 1)` with no sigmoid; critic loss `C(fake).mean() - C(real).mean() + 10 * penalty`, where the penalty is $(\lVert \nabla_{\hat x} C(\hat x) \rVert - 1)^2$ averaged over random points $\hat x$ on lines between real and fake samples; five critic steps per generator step; generator loss `-C(G(z)).mean()`; Adam with `lr=1e-3, betas=(0.5, 0.9)`.
3. Train it for 1,500 generator steps from seed 8, print `mode_report` every 250 steps, and compare all three runs. Which fix covers the modes, and which makes the sharpest samples?

In [ ]:
# Your code here

## Normalising flows

### The change of variables formula

If an invertible $f$ maps data to noise, $z = f(x) \sim \mathcal{N}(0, I)$, then

$$\log p_X(x) = \log p_Z\big(f(x)\big) + \log\left|\det \frac{\partial f(x)}{\partial x}\right|.$$

### An affine coupling layer

Keep half of the input, scale and shift the other half by amounts computed from the first: $z_b = b \odot e^{s(a)} + t(a)$, with $\log|\det J| = \sum s(a)$. The inverse is just as cheap.

In [ ]:
class Coupling(nn.Module):
    """Affine coupling: keep one coordinate, scale and shift the other."""
    def __init__(self, flip):
        super().__init__()
        self.flip = flip
        self.net = mlp(1, 2, hidden=64)      # outputs log-scale s and shift t

    def split(self, x):
        a, b = x[:, :1], x[:, 1:]
        return (b, a) if self.flip else (a, b)

    def join(self, a, b):
        return torch.cat([b, a] if self.flip else [a, b], dim=1)

    def forward(self, x):                    # data -> noise, with log|det J|
        a, b = self.split(x)
        s, t = self.net(a).chunk(2, dim=1)
        s = torch.tanh(s)                    # keep the scale factors tame
        return self.join(a, b * torch.exp(s) + t), s.squeeze(1)

    def inverse(self, z):                    # noise -> data
        a, b = self.split(z)
        s, t = self.net(a).chunk(2, dim=1)
        s = torch.tanh(s)
        return self.join(a, (b - t) * torch.exp(-s))

class Flow(nn.Module):
    def __init__(self, n_layers=8):
        super().__init__()
        self.layers = nn.ModuleList(Coupling(flip=i % 2 == 1)
                                    for i in range(n_layers))

    def forward(self, x):                    # data -> noise, summing log|det J|
        log_det = torch.zeros(len(x))
        for layer in self.layers:
            x, ld = layer(x)
            log_det = log_det + ld
        return x, log_det

    def inverse(self, z):                    # noise -> data
        for layer in reversed(self.layers):
            z = layer.inverse(z)
        return z

    def log_prob(self, x):                   # the change of variables formula
        z, log_det = self(x)
        return -0.5 * (z ** 2).sum(1) - np.log(2 * np.pi) + log_det

M = torch.tensor(moons, dtype=torch.float32)
torch.manual_seed(0)
flow = Flow()
opt = torch.optim.Adam(flow.parameters(), lr=2e-3)
for step in range(2000):
    xb = M[:1500][torch.randint(0, 1500, (256,))]
    loss = -flow.log_prob(xb).mean()            # maximum likelihood
    opt.zero_grad(); loss.backward(); opt.step()

with torch.no_grad():
    print("flow test log p(x):", round(flow.log_prob(M[1500:]).mean().item(), 3))
    print("GMM  test log p(x):", round(gmm.score(moons[1500:]), 3))
    z, _ = flow(M[1500:1505])
    print("round trip:", (flow.inverse(z) - M[1500:1505]).abs().max().item())

In [ ]:
torch.manual_seed(0)
with torch.no_grad():
    flow_samples = flow.inverse(torch.randn(1000, 2))
show_points([M[1500:], flow_samples], ["held-out data", "flow samples"], "A coupling flow on two moons")

The flow beats the Gaussian mixture from session 1 on the same held-out points, and the round trip is exact up to float rounding.

## Diffusion from scratch

Lesson 23 gave the intuition and the forward process $x_t = \sqrt{\bar\alpha_t}\,x_0 + \sqrt{1-\bar\alpha_t}\,\varepsilon$. Ancestral sampling runs it backwards:

$$x_{t-1} = \frac{1}{\sqrt{\alpha_t}}\left(x_t - \frac{\beta_t}{\sqrt{1-\bar\alpha_t}}\,\varepsilon_\theta(x_t, t)\right) + \sigma_t\,\mathbf{n},$$

with $\sigma_t^2 = \tilde\beta_t = \beta_t(1-\bar\alpha_{t-1})/(1-\bar\alpha_t)$, which gives sharper samples than $\beta_t$ on this tiny dataset.

In [ ]:
T = 100
betas = torch.linspace(1e-4, 0.2, T)   # linear schedule, rescaled for 100 steps
alphas = 1 - betas
alpha_bar = torch.cumprod(alphas, dim=0)
alpha_bar_prev = torch.cat([torch.ones(1), alpha_bar[:-1]])
sigma2 = betas * (1 - alpha_bar_prev) / (1 - alpha_bar)    # posterior variance

torch.manual_seed(0)
eps_model = mlp(3, 2, depth=3)          # (x_t, t/T) -> predicted noise
opt = torch.optim.Adam(eps_model.parameters(), lr=1e-3)
for step in range(4000):
    x0 = sample_ring(256)
    t = torch.randint(0, T, (256,))
    eps = torch.randn_like(x0)
    ab = alpha_bar[t].unsqueeze(1)
    xt = ab.sqrt() * x0 + (1 - ab).sqrt() * eps            # forward process
    pred = eps_model(torch.cat([xt, t.unsqueeze(1) / T], dim=1))
    loss = F.mse_loss(pred, eps)
    opt.zero_grad(); loss.backward(); opt.step()

@torch.no_grad()
def ddpm_sample(n):
    x = torch.randn(n, 2)                              # start from pure noise
    for t in reversed(range(T)):
        eps = eps_model(torch.cat([x, torch.full((n, 1), t / T)], dim=1))
        x = (x - betas[t] / (1 - alpha_bar[t]).sqrt() * eps) / alphas[t].sqrt()
        if t > 0:
            x = x + sigma2[t].sqrt() * torch.randn_like(x)  # fresh noise
    return x

torch.manual_seed(1)
print(mode_report(ddpm_sample(1000)))

In [ ]:
torch.manual_seed(0)
show_points([sample_ring(1000), ddpm_sample(1000)], ["data", "DDPM samples"], "DDPM covers every mode")

All eight modes with a plain regression loss and no adversary. The price is 100 network calls per sample.

## Flow matching and rectified flow

Take the straight line from noise $x_0$ to data $x_1$, $x_t = (1-t)\,x_0 + t\,x_1$, and regress the velocity $x_1 - x_0$. Generate by integrating $dx/dt = v_\theta(x, t)$ from $t = 0$ to $1$ with Euler steps.

In [ ]:
torch.manual_seed(0)
v_model = mlp(3, 2, depth=3)            # (x_t, t) -> velocity
opt = torch.optim.Adam(v_model.parameters(), lr=1e-3)
for step in range(4000):
    x1 = sample_ring(256)                        # data
    x0 = torch.randn_like(x1)                    # noise
    t = torch.rand(256, 1)
    xt = (1 - t) * x0 + t * x1                   # a point on the straight line
    loss = F.mse_loss(v_model(torch.cat([xt, t], dim=1)), x1 - x0)
    opt.zero_grad(); loss.backward(); opt.step()

@torch.no_grad()
def fm_sample(n, n_steps):
    x = torch.randn(n, 2)
    for i in range(n_steps):                     # Euler steps from t=0 to t=1
        t = torch.full((n, 1), i / n_steps)
        x = x + v_model(torch.cat([x, t], dim=1)) / n_steps
    return x

torch.manual_seed(1)
for n_steps in [1, 5, 20, 100]:
    print(n_steps, mode_report(fm_sample(1000, n_steps)))

In [ ]:
torch.manual_seed(0)
fig, axes = plt.subplots(1, 3, figsize=(11, 3.8))
for ax, n_steps in zip(axes, [1, 5, 20]):
    pts = fm_sample(1000, n_steps).numpy()
    ax.scatter(pts[:, 0], pts[:, 1], s=3, alpha=0.5)
    ax.set_title(f"{n_steps} Euler step(s)")
    ax.set_xlim(-3, 3); ax.set_ylim(-3, 3); ax.set_aspect("equal")
plt.show()

One step sends every sample to the middle of the ring (the average destination); a handful of steps is enough after that.

## Latent diffusion and autoregressive models

Latent diffusion runs diffusion or flow matching in an autoencoder's latent space; the last exercise builds a tiny one. Autoregressive models (lesson 08) are generative models too: $p(x) = \prod_i p(x_i \mid x_{<i})$ gives an exact likelihood, and sampling token by token is ancestral sampling.

## Evaluating generative models

Check realism, variety and novelty separately. Below: a Fréchet distance on digits (FID's formula, with features from a small classifier instead of Inception), including the small-sample bias, and k-nearest-neighbour precision and recall on the ring.

In [ ]:
from scipy import linalg
from sklearn.neighbors import NearestNeighbors
from sklearn.neural_network import MLPClassifier

# 1. a Frechet distance on digits, with features from a small classifier
feat_net = MLPClassifier((64,), max_iter=1000, random_state=0).fit(X_tr, y_tr)
def features(x):
    return np.maximum(0, x @ feat_net.coefs_[0] + feat_net.intercepts_[0])

def frechet(a, b):
    fa, fb = features(a), features(b)
    mu_a, mu_b = fa.mean(0), fb.mean(0)
    eps = 1e-6 * np.eye(fa.shape[1])           # keeps sqrtm numerically stable
    cov_a = np.cov(fa, rowvar=False) + eps
    cov_b = np.cov(fb, rowvar=False) + eps
    covmean = linalg.sqrtm(cov_a @ cov_b).real
    return ((mu_a - mu_b) ** 2).sum() + np.trace(cov_a + cov_b - 2 * covmean)

print("real vs real    ", round(frechet(X_te, X_tr), 2))
print("VAE samples     ", round(frechet(X_te, samples), 2))
print("AE, N(0,I) codes", round(frechet(X_te, fake), 2))
print("real, only 50   ", round(frechet(X_te[:50], X_tr[:50]), 2))

# 2. precision and recall on the ring (k-nearest-neighbour balls)
def precision_recall(real, fake, k=5):
    def radii(a):          # distance from each point to its k-th neighbour
        knn = NearestNeighbors(n_neighbors=k + 1).fit(a)
        return torch.tensor(knn.kneighbors(a)[0][:, -1])
    def inside(a, b):      # share of a that falls in some ball around b
        return (torch.cdist(a, b) <= radii(b)).any(1).float().mean().item()
    return round(inside(fake, real), 2), round(inside(real, fake), 2)

torch.manual_seed(2)
real = sample_ring(1000)
with torch.no_grad():
    gan_fake = G(torch.randn(1000, 2))
print("GAN  (precision, recall):", precision_recall(real, gan_fake))
print("DDPM (precision, recall):", precision_recall(real, ddpm_sample(1000)))

Similar precision for the GAN and the diffusion model, very different recall: recall is the number that exposes mode collapse. For memorisation risks, see lesson 22.

### Hands-on: build a tiny latent diffusion model

1. Train `AutoEncoder(d_latent=8)` with `fit`, encode the training digits, and standardise the codes (subtract the mean, divide by the standard deviation).
2. Train flow matching in that 8-D latent space: `mlp(9, 8, hidden=256, depth=3)`, 3,000 steps, batch 256.
3. Generate 1,000 latents with 50 Euler steps, undo the standardisation and decode. Report `judge_report` and `frechet` against `X_te`. Compare with decoding Gaussian codes that have the right mean and standard deviation but no learnt model.
4. Check for memorisation: compare the median distance from each generated digit to its nearest training digit with the same figure for held-out real digits.

You are done when your latent model beats the VAE's Fréchet distance of 7.07 and covers all ten digits.

In [ ]:
# Your code here

## Check yourself

Take the quiz at the end of [the lesson page](../lessons/26-generative-models.html). Then close everything and write three or four sentences in your learning log about what you learned, from memory.